# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [2]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [3]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: /home/ahs/Personal Projects/mlops-bootcamp/week2-01
FEATURE_DIR: /home/ahs/Personal Projects/mlops-bootcamp/week2-01/data/features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [5]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)
from dotenv import load_dotenv

load_dotenv()
DB_HOST = os.getenv("PGHOST", "")
DB_PORT = int(os.getenv("PGPORT", ""))
DB_NAME = os.getenv("PGDATABASE", "")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")


if DB_USER == "student_your_username" or DB_PASSWORD == "student_your_password":
    raise ValueError("Replace DB_USER and DB_PASSWORD with your assigned database credentials.")

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url, pool_pre_ping=True)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)

Connecting to:
HOST: 185.50.38.163
PORT: 32112
DB: qbc12_airbnb
USER: student_amirhossein_sa


{'database': 'qbc12_airbnb',
 'user_name': 'student_amirhossein_sa',
 'server_ip': '172.19.0.9',
 'server_port': 5432,
 'checked_at': datetime.datetime(2026, 6, 21, 11, 29, 4, 214542, tzinfo=datetime.timezone.utc)}

## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [6]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

,table_schema,table_name,table_type
0,core,calendar_day,BASE TABLE
1,core,host,BASE TABLE
2,core,listing,BASE TABLE
3,core,neighbourhood,BASE TABLE
4,core,review,BASE TABLE


In [7]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,core,calendar_day,1,listing_id,bigint,NO
1,core,calendar_day,2,date,date,NO
2,core,calendar_day,3,available,boolean,YES
3,core,calendar_day,4,price,numeric,YES
4,core,calendar_day,5,adjusted_price,numeric,YES
5,core,calendar_day,6,minimum_nights,integer,YES
6,core,calendar_day,7,maximum_nights,integer,YES
7,core,host,1,host_id,bigint,NO
8,core,host,2,host_pseudo_id,text,NO
9,core,host,3,is_superhost,boolean,YES


In [8]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

,table_name,row_count
0,core.calendar_day,3825200
1,core.host,9201
2,core.listing,10480
3,core.neighbourhood,22
4,core.review,501084


## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [9]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

,n_rows,null_price,null_adjusted_price,null_available,min_calendar_date,max_calendar_date
0,3825200,3825200,3825200,0,2025-09-11,2026-09-10


,n_rows,null_comment_len,min_review_date,max_review_date
0,501084,0,2010-08-22,2025-09-11


In [10]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))


===== core.listing =====


,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,license
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,0363 974D 4986 7411 88D8
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,0363 607B EA74 0BD8 2F6F
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,0363 607B EA74 0BD8 2F6F
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,0363 E76E F06A C1DD 172C
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,0363 4A2B A6AD 0196 F684
5,49552,225987,2,Entire home/apt,Entire guest suite,3,2.0,2.0,1 bath,322.0,3,1125,False,0363 576A D827 5085 6B83
6,50263,230246,1,Entire home/apt,Entire condo,4,2.0,3.0,1.5 baths,457.0,2,14,False,0363 7F3D 0BAE 28C8 C7D2
7,50515,231864,14,Entire home/apt,Entire townhouse,5,3.0,3.0,1.5 baths,198.0,7,18,False,0363 5DDB E495 A6D5 CEC6
8,50523,231946,2,Entire home/apt,Entire condo,2,1.0,1.0,1 bath,162.0,2,365,False,0363 22DC 0E52 B70B 0FB8
9,53921,252245,10,Entire home/apt,Entire rental unit,3,1.0,NaN,1 bath,NaN,1,21,False,0363 B43C B1D4 2666 3739



===== core.host =====


,host_id,host_pseudo_id,is_superhost
0,27837566,12a252de05fbf2f7ba9f57fa3baa099acd17e2a9c7efc7...,False
1,12840373,d9f7e79668b99a5cb7963bfb2430d8b6b960a0ec4e82bb...,False
2,226859324,cc90d30412e286c0525c7914d031807c8c00e017fe1915...,True
3,20204265,755d79bf4be51df2d85792123200e6641db8589551965e...,True
4,47981094,b40c45156e81696746b6eb51ef86aeccff7c6d7cd5eac2...,False
5,443406859,cfe69dd56cffef559069b30216f8954b57e1de886b92a3...,False
6,2626085,43c4cdd7f1413364dd809c6c92c20baed35faa51f84e76...,False
7,74999205,1033d2369452b0969c1f63061af401f4581a92873b5659...,True
8,7969106,1b41831025e74b04a73fe88e99233d09f39660eac85bf6...,False
9,6127483,8cffc835ea5e24b102cf446e9d369edad9f3a2bf91bfbc...,True



===== core.neighbourhood =====


,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart
5,6,Bijlmer-Centrum
6,7,Geuzenveld - Slotermeer
7,8,Buitenveldert - Zuidas
8,9,Noord-West
9,10,IJburg - Zeeburgereiland



===== core.review =====


,review_id,listing_id,review_date,reviewer_id,reviewer_pseudo_id,comment_len
0,33387684,839610,2015-05-27,32412055,fea5935e3a897d4e1355404122b2a0b97c9792aa8a2d82...,64
1,33865462,839610,2015-06-01,33521461,dcc5a1a3bb361c9ef351bcfd0377d977fc3fd3aebbac49...,37
2,47062612,839610,2015-09-15,25521258,b702f45abf3cac2e595b7e2748d7ca7990b2e51739e072...,380
3,49847491,839610,2015-10-06,32115691,49484619e10bcd79688956e49d10be722924748fec936f...,111
4,62280485,839610,2016-02-13,16551781,8718259d45baff7b21f21175cfe4420e290762f6db5d22...,69
5,65253023,839610,2016-03-12,15017175,9901e07d216722a8f4ff7178b18856ad5b32dd558b66c6...,291
6,66472891,839610,2016-03-22,10295650,fd221edd94dd0fa06dccdbcb9fb6518e088dd2a0845039...,112
7,209427303,839610,2017-11-05,155615987,87ad4e2d2752505fe72495c996d5f7ee6fae1b628cc378...,143
8,210242042,839610,2017-11-08,145039581,088d50bb9579a3e25ca472c6920f59a644b7935bf3fc7f...,311
9,211800819,839610,2017-11-14,120854205,4280772cf54f1a7156117739258e54ecce626d004248a0...,218



===== core.calendar_day =====


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,538723,2025-09-11,False,None,None,5,30
1,538723,2025-09-12,False,None,None,5,30
2,538723,2025-09-13,False,None,None,5,30
3,538723,2025-09-14,False,None,None,5,30
4,538723,2025-09-15,False,None,None,5,30
5,538723,2025-09-16,False,None,None,5,30
6,538723,2025-09-17,False,None,None,5,30
7,538723,2025-09-18,False,None,None,5,30
8,538723,2025-09-19,False,None,None,5,30
9,538723,2025-09-20,False,None,None,5,30


## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [11]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

,calendar_min_date,calendar_max_date,review_min_date,review_max_date
0,2025-09-11,2026-09-10,2010-08-22,2025-09-11


In [12]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

earliest_cutoff_allowed_by_calendar = calendar_min_date + timedelta(days=PAST_WINDOW_DAYS)
latest_cutoff_allowed_by_calendar = calendar_max_date - timedelta(days=FUTURE_WINDOW_DAYS)

# Use the earliest valid cutoff to keep review recency as meaningful as possible.
cutoff_date = earliest_cutoff_allowed_by_calendar
history_start_date = cutoff_date - timedelta(days=PAST_WINDOW_DAYS)
label_end_date = cutoff_date + timedelta(days=FUTURE_WINDOW_DAYS)

print("calendar_min_date:", calendar_min_date)
print("calendar_max_date:", calendar_max_date)
print("earliest_cutoff_allowed_by_calendar:", earliest_cutoff_allowed_by_calendar)
print("latest_cutoff_allowed_by_calendar:", latest_cutoff_allowed_by_calendar)
print("chosen cutoff_date:", cutoff_date)
print("history_start_date:", history_start_date)
print("label_end_date:", label_end_date)

assert history_start_date >= calendar_min_date
assert label_end_date <= calendar_max_date
assert earliest_cutoff_allowed_by_calendar <= cutoff_date <= latest_cutoff_allowed_by_calendar

calendar_min_date: 2025-09-11
calendar_max_date: 2026-09-10
earliest_cutoff_allowed_by_calendar: 2025-12-10
latest_cutoff_allowed_by_calendar: 2026-08-11
chosen cutoff_date: 2025-12-10
history_start_date: 2025-09-11
label_end_date: 2026-01-09


## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [13]:
pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input",
    },
    {
        "table": "listing",
        "column": "host_id",
        "pii_type": "identity-linking identifier",
        "decision": "use only for joins and host-level aggregates; exclude from final dataset",
        "reason": "links listings to a specific host identity",
    },
    {
        "table": "host",
        "column": "host_pseudo_id",
        "pii_type": "pseudonymous identifier",
        "decision": "exclude from final dataset",
        "reason": "persistent identifier that can link records to one host",
    },
    {
        "table": "listing",
        "column": "license",
        "pii_type": "license / registration identifier",
        "decision": "exclude from final dataset",
        "reason": "identity-linking administrative identifier and mostly not useful as a model feature",
    },
    {
        "table": "review",
        "column": "review_id",
        "pii_type": "event identifier",
        "decision": "exclude from final dataset",
        "reason": "raw review identifier is not a stable ML feature",
    },
    {
        "table": "review",
        "column": "reviewer_id",
        "pii_type": "identity-linking identifier",
        "decision": "aggregate only; exclude raw values from final dataset",
        "reason": "links reviews to a specific reviewer identity",
    },
    {
        "table": "review",
        "column": "reviewer_pseudo_id",
        "pii_type": "pseudonymous identifier",
        "decision": "aggregate only; exclude raw values from final dataset",
        "reason": "persistent reviewer identifier can link records across reviews",
    },
    {
        "table": "review",
        "column": "raw review text",
        "pii_type": "free text",
        "decision": "do not use raw text; use only comment_len aggregate already in core.review",
        "reason": "free text can contain names, contact details, or other sensitive information",
    },
])

pii_audit

,table,column,pii_type,decision,reason
0,listing,listing_id,entity identifier,keep as entity key only,needed to define one row per listing; not a mo...
1,listing,host_id,identity-linking identifier,use only for joins and host-level aggregates; ...,links listings to a specific host identity
2,host,host_pseudo_id,pseudonymous identifier,exclude from final dataset,persistent identifier that can link records to...
3,listing,license,license / registration identifier,exclude from final dataset,identity-linking administrative identifier and...
4,review,review_id,event identifier,exclude from final dataset,raw review identifier is not a stable ML feature
5,review,reviewer_id,identity-linking identifier,aggregate only; exclude raw values from final ...,links reviews to a specific reviewer identity
6,review,reviewer_pseudo_id,pseudonymous identifier,aggregate only; exclude raw values from final ...,persistent reviewer identifier can link record...
7,review,raw review text,free text,do not use raw text; use only comment_len aggr...,"free text can contain names, contact details, ..."


## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [14]:
listing_df = read_sql("""
SELECT
    listing_id,
    host_id,
    neighbourhood_id,
    room_type,
    property_type,
    accommodates,
    bedrooms,
    beds,
    bathrooms_text,
    listing_price,
    minimum_nights,
    maximum_nights,
    instant_bookable
FROM core.listing;
""")

host_df = read_sql("""
SELECT
    host_id,
    is_superhost AS host_is_superhost
FROM core.host;
""")

neighbourhood_df = read_sql("""
SELECT
    neighbourhood_id,
    name AS neighbourhood_name
FROM core.neighbourhood;
""")

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

listing: (10480, 13)
host: (9201, 2)
neighbourhood: (22, 2)


## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [15]:
def normalize_boolean(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")

    return (
        series
        .map({
            True: True,
            False: False,
            "t": True,
            "f": False,
            "true": True,
            "false": False,
            "True": True,
            "False": False,
            1: True,
            0: False,
        })
        .astype("boolean")
    )


listing_df["instant_bookable"] = normalize_boolean(listing_df["instant_bookable"])
host_df["host_is_superhost"] = normalize_boolean(host_df["host_is_superhost"])

numeric_cols_listing = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in numeric_cols_listing:
    listing_df[col] = pd.to_numeric(listing_df[col], errors="coerce")


def parse_bathrooms(text_value):
    """
    Convert bathrooms_text into a number.

    Examples:
    - '1 bath' -> 1.0
    - '1.5 baths' -> 1.5
    - 'Half-bath' -> 0.5
    - missing/unrecognized -> NaN
    """
    if pd.isna(text_value):
        return np.nan

    text_value = str(text_value).strip().lower()
    if not text_value:
        return np.nan

    if "half" in text_value:
        return 0.5

    match = re.search(r"(\d+(?:\.\d+)?)", text_value)
    if match:
        return float(match.group(1))

    return np.nan


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)

listing_df[["bathrooms_text", "bathrooms"]].head(10)

,bathrooms_text,bathrooms
0,1.5 baths,1.5
1,1 shared bath,1.0
2,1 shared bath,1.0
3,1.5 baths,1.5
4,1.5 baths,1.5
5,1 bath,1.0
6,1.5 baths,1.5
7,1.5 baths,1.5
8,1 bath,1.0
9,1 bath,1.0


## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [16]:
host_listing_features = (
    listing_df
    .groupby("host_id", as_index=False)
    .size()
    .rename(columns={"size": "host_listing_count"})
)

base_listing_features = (
    listing_df
    .merge(host_df, on="host_id", how="left")
    .merge(host_listing_features, on="host_id", how="left")
    .merge(neighbourhood_df, on="neighbourhood_id", how="left")
)

static_feature_cols = [
    "listing_id",
    "room_type",
    "property_type",
    "neighbourhood_name",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "host_is_superhost",
    "host_listing_count",
]

static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()

(10480, 14)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,instant_bookable,host_is_superhost,host_listing_count
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,356,False,True,1
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,730,False,True,2
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,730,False,True,2
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,730,False,False,1
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,1125,False,False,1


## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [17]:
review_features = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*)::integer AS total_reviews_before_cutoff,
        COUNT(DISTINCT reviewer_pseudo_id)::integer AS unique_reviewers_before_cutoff,
        AVG(comment_len)::double precision AS avg_comment_len_before_cutoff,
        MAX(comment_len)::integer AS max_comment_len_before_cutoff,
        (CAST(:cutoff_date AS date) - MAX(review_date))::integer AS days_since_last_review
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

review_numeric_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
    "days_since_last_review",
]

for col in review_numeric_cols:
    review_features[col] = pd.to_numeric(review_features[col], errors="coerce")

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
review_features.head()

(9383, 6)


,listing_id,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review
0,27886,311,311,302.167203,1917,94
1,28871,732,729,201.236339,1265,94
2,29051,849,841,245.108363,2253,93
3,44391,42,42,242.309524,891,1208
4,48373,5,5,272.200000,949,591


## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [18]:
calendar_features_all = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*)::integer AS calendar_days_observed_last_90d,
        SUM(available::integer)::integer AS available_days_last_90d,
        AVG(available::integer)::double precision AS available_rate_last_90d,
        AVG(minimum_nights)::double precision AS avg_minimum_nights_calendar_last_90d,
        AVG(maximum_nights)::double precision AS avg_maximum_nights_calendar_last_90d,
        COUNT(*) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        )::integer AS calendar_days_observed_last_30d,
        SUM(available::integer) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        )::integer AS available_days_last_30d,
        AVG(available::integer) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        )::double precision AS available_rate_last_30d,
        AVG(minimum_nights) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        )::double precision AS avg_minimum_nights_calendar_last_30d,
        AVG(maximum_nights) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        )::double precision AS avg_maximum_nights_calendar_last_30d
    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "history_start_date": history_start_date,
        "cutoff_date": cutoff_date,
    },
)

calendar_numeric_cols = [
    col for col in calendar_features_all.columns
    if col != "listing_id"
]

for col in calendar_numeric_cols:
    calendar_features_all[col] = pd.to_numeric(calendar_features_all[col], errors="coerce")

assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
calendar_features_all.head()

(10480, 11)


,listing_id,calendar_days_observed_last_90d,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,calendar_days_observed_last_30d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d
0,1443670960781261954,91,0,0.000000,2.000000,30.0,30,0,0.0,2.0,30.0
1,896043282611946316,91,0,0.000000,5.000000,25.0,30,0,0.0,5.0,25.0
2,39969190,91,0,0.000000,3.000000,9.0,30,0,0.0,3.0,9.0
3,958726726744532841,91,1,0.010989,1.989011,7.0,30,0,0.0,2.0,7.0
4,1093563123501570178,91,3,0.032967,2.967033,30.0,30,3,0.1,3.0,30.0


## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [19]:
label_df = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*)::integer AS future_calendar_days_observed_30d,
        SUM(available::integer)::integer AS future_available_days_30d,
        AVG(available::integer)::double precision AS future_available_rate_30d,
        CASE
            WHEN AVG(available::integer) <= :threshold THEN 1
            ELSE 0
        END::integer AS high_demand_proxy
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
)

label_numeric_cols = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

for col in label_numeric_cols:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce")

label_df["high_demand_proxy"] = label_df["high_demand_proxy"].astype("int64")

assert label_df["listing_id"].duplicated().sum() == 0

print(label_df.shape)
label_df.head()

(10480, 5)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy
0,1443670960781261954,30,0,0.000000,1
1,896043282611946316,30,0,0.000000,1
2,958726726744532841,30,0,0.000000,1
3,39969190,30,0,0.000000,1
4,1476051889347548382,30,23,0.766667,0


In [20]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

,high_demand_proxy,count,percentage
0,1,7005,0.6684
1,0,3475,0.3316


## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [21]:
feature_df = (
    static_features
    .merge(review_features, on="listing_id", how="left")
    .merge(calendar_features_all, on="listing_id", how="inner")
    .merge(label_df, on="listing_id", how="inner")
)

feature_df.insert(1, "cutoff_date", str(cutoff_date))
feature_df.insert(2, "dataset_version", DATASET_VERSION)

review_count_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
]
review_summary_cols = [
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
]

for col in review_count_cols:
    feature_df[col] = feature_df[col].fillna(0).astype("int64")

for col in review_summary_cols:
    feature_df[col] = feature_df[col].fillna(0)

review_min_date = pd.to_datetime(range_df.loc[0, "review_min_date"]).date()
no_review_days_since_sentinel = (cutoff_date - review_min_date).days + 1

feature_df["has_reviews_before_cutoff"] = (
    feature_df["total_reviews_before_cutoff"] > 0
).astype("int64")

feature_df["days_since_last_review"] = (
    feature_df["days_since_last_review"]
    .fillna(no_review_days_since_sentinel)
    .astype("int64")
)

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
feature_df.head()

(10480, 36)


,listing_id,cutoff_date,dataset_version,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,...,calendar_days_observed_last_30d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,has_reviews_before_cutoff
0,27886,2025-12-10,v1_student,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,...,30,10,0.333333,3.000000,30.0,30,1,0.033333,1,1
1,28871,2025-12-10,v1_student,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,...,30,8,0.266667,1.866667,730.0,30,4,0.133333,1,1
2,29051,2025-12-10,v1_student,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,...,30,12,0.400000,2.000000,730.0,30,0,0.000000,1,1
3,44391,2025-12-10,v1_student,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,...,30,0,0.000000,3.000000,730.0,30,0,0.000000,1,1
4,48373,2025-12-10,v1_student,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,...,30,0,0.000000,3.000000,1125.0,30,0,0.000000,1,1


## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [22]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

missing_rates = feature_df.isna().mean()

high_missing_columns = [
    col for col, missing_rate in missing_rates.items()
    if col not in protected_columns and missing_rate > HIGH_MISSING_DROP_THRESHOLD
]

constant_columns = [
    col for col in feature_df.columns
    if col not in protected_columns and feature_df[col].nunique(dropna=False) <= 1
]

columns_to_drop = sorted(set(high_missing_columns + constant_columns))

print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)

Columns to drop: ['calendar_days_observed_last_30d', 'calendar_days_observed_last_90d']
New shape: (10480, 34)


## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [23]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(
    int(value) for value in feature_df["high_demand_proxy"].dropna().unique().tolist()
)

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

assert duplicate_count == 0
assert missing_target_count == 0
assert set(unique_target_values).issubset({0, 1})
assert len(unique_target_values) == 2
assert present_forbidden_columns == []
assert future_leakage_columns == []
assert len(model_input_columns) > 0

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))

duplicate_count: 0
missing_target_count: 0
unique_target_values: [0, 1]
present_forbidden_columns: []
future_leakage_columns: []
model_input_column_count: 27


In [24]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

,column,missing_rate
0,listing_price,0.439504
1,beds,0.436641
2,bedrooms,0.029198
3,host_is_superhost,0.010973
4,bathrooms,0.001145
5,listing_id,0.000000
6,cutoff_date,0.000000
7,property_type,0.000000
8,neighbourhood_name,0.000000
9,dataset_version,0.000000


,high_demand_proxy,count,percentage
0,1,7005,0.6684
1,0,3475,0.3316


,index,future_calendar_days_observed_30d
0,count,10480.0
1,mean,30.0
2,std,0.0
3,min,30.0
4,25%,30.0
5,50%,30.0
6,75%,30.0
7,max,30.0


## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [25]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")


def dataframe_records(df: pd.DataFrame) -> list[dict]:
    """Convert a DataFrame to JSON-safe Python records."""
    return json.loads(df.to_json(orient="records"))


metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    "cutoff_date": str(cutoff_date),
    "history_start_date": str(history_start_date),
    "label_end_date": str(label_end_date),
    "past_window_days": PAST_WINDOW_DAYS,
    "future_window_days": FUTURE_WINDOW_DAYS,
    "source_tables": [
        "core.listing",
        "core.host",
        "core.neighbourhood",
        "core.review",
        "core.calendar_day",
    ],
    "target_definition": {
        "target_column": "high_demand_proxy",
        "positive_class_rule": (
            "1 when future_available_rate_30d <= "
            f"{HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD}; else 0"
        ),
        "label_columns": label_only_columns,
    },
    "privacy_rules": {
        "entity_key": "listing_id is retained only as an entity key",
        "excluded_raw_columns": sorted(forbidden_columns - {"bathrooms_text"}),
        "raw_text_policy": "raw review text is not used; only comment_len aggregates are used",
    },
    "data_quality_notes": {
        "calendar_price_features": "excluded because calendar_day.price and adjusted_price are all null in the audit",
        "no_review_days_since_last_review_sentinel": int(no_review_days_since_sentinel),
        "dropped_columns": columns_to_drop,
    },
    "row_count": int(feature_df.shape[0]),
    "column_count": int(feature_df.shape[1]),
    "model_input_columns": model_input_columns,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": unique_target_values,
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    "model_input_column_count": len(model_input_columns),
    "missing_report": dataframe_records(missing_report),
    "label_distribution": dataframe_records(label_distribution),
    "calendar_coverage_summary": dataframe_records(calendar_coverage_summary),
    "columns_to_drop": columns_to_drop,
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

Saved CSV: /home/ahs/Personal Projects/mlops-bootcamp/week2-01/data/features/listing_availability_features_v1_student.csv
Saved Parquet: /home/ahs/Personal Projects/mlops-bootcamp/week2-01/data/features/listing_availability_features_v1_student.parquet
Saved metadata: /home/ahs/Personal Projects/mlops-bootcamp/week2-01/data/features/listing_availability_features_v1_student_metadata.json
Saved validation report: /home/ahs/Personal Projects/mlops-bootcamp/week2-01/data/features/listing_availability_features_v1_student_validation_report.json
Saved PII audit: /home/ahs/Personal Projects/mlops-bootcamp/week2-01/data/features/pii_audit_v1_student.csv


## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [26]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()

Final shape: (10480, 34)


,listing_id,cutoff_date,dataset_version,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,...,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,has_reviews_before_cutoff
0,27886,2025-12-10,v1_student,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,...,30.0,10,0.333333,3.000000,30.0,30,1,0.033333,1,1
1,28871,2025-12-10,v1_student,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,...,730.0,8,0.266667,1.866667,730.0,30,4,0.133333,1,1
2,29051,2025-12-10,v1_student,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,...,730.0,12,0.400000,2.000000,730.0,30,0,0.000000,1,1
3,44391,2025-12-10,v1_student,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,...,730.0,0,0.000000,3.000000,730.0,30,0,0.000000,1,1
4,48373,2025-12-10,v1_student,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,...,1125.0,0,0.000000,3.000000,1125.0,30,0,0.000000,1,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10480 entries, 0 to 10479
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   listing_id                            10480 non-null  int64  
 1   cutoff_date                           10480 non-null  object 
 2   dataset_version                       10480 non-null  object 
 3   room_type                             10480 non-null  object 
 4   property_type                         10480 non-null  object 
 5   neighbourhood_name                    10480 non-null  object 
 6   accommodates                          10480 non-null  int64  
 7   bedrooms                              10174 non-null  float64
 8   beds                                  5904 non-null   float64
 9   bathrooms                             10468 non-null  float64
 10  listing_price                         5874 non-null   float64
 11  minimum_nights 